In [2]:
!pip install gradio transformers torch

In [4]:
# Menú de La Parada amb preus i estoc
MENU = {
    "hamburguesa bbq": {"nom": "Hamburguesa BBQ", "preu": 8.50, "estoc": 12},
    "hamburguesa clasica": {"nom": "Hamburguesa Clàssica", "preu": 7.50, "estoc": 10},
    "sandvitx pulled pork": {"nom": "Sandvitx Pulled Pork", "preu": 7.00, "estoc": 8},
    "sandvitx escalivada": {"nom": "Sandvitx Escalivada", "preu": 6.50, "estoc": 6},
    "patates": {"nom": "Patates Fregides", "preu": 3.00, "estoc": 20},
    "beguda natural taronja": {"nom": "Beguda Natural Taronja", "preu": 3.00, "estoc": 15},
    "beguda natural llimona": {"nom": "Beguda Natural Llimona", "preu": 3.00, "estoc": 15},
    "batut maduixa": {"nom": "Batut Maduixa", "preu": 3.50, "estoc": 10},
}

print("Menú de La Parada carregat")
print(f"Total de productes: {len(MENU)}")

Menú de La Parada carregat
Total de productes: 8


In [5]:
from transformers import pipeline

# Carreguem el model de chatbot (gratuït, es descarrega automàticament)
print("Carregant model d'IA... (pot trigar una estona)")

chatbot = pipeline("text-generation", model="microsoft/DialoGPT-medium")

print("Model d'IA carregat!")

Carregant model d'IA... (pot trigar una estona)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/863M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Model d'IA carregat!


In [6]:
def processar_comanda(missatge, historial):
    # Buscar productes del menú a la comanda
    comanda_detectada = []
    total = 0.0
    missatge_lower = missatge.lower()

    for clau, producte in MENU.items():
        if any(paraula in missatge_lower for paraula in clau.split()):
            if producte["estoc"] > 0:
                comanda_detectada.append(producte)
                total += producte["preu"]
                # Reduïm estoc
                MENU[clau]["estoc"] -= 1

    if comanda_detectada:
        llista = "\n".join([f"  • {p['nom']} — {p['preu']}€" for p in comanda_detectada])
        resposta = f"Comanda registrada!\n\n{llista}\n\n💰 Total: {total:.2f}€\n Temps d'espera: ~10 minuts"
    else:
        resposta = "Hola! Soc l'assistent de La Parada. Pots demanar:\n Hamburguesa BBQ (8.50€)\n Sandvitx Pulled Pork (7€)\n Patates (3€)\n Begudes naturals (3€)"

    return resposta

print("Funció de comandes llesta!")

Funció de comandes llesta!


In [7]:
import gradio as gr

with gr.Blocks(title="La Parada - Assistent de Comandes") as demo:
    gr.Markdown("# 🍔 La Parada — Assistent de Comandes")
    gr.Markdown("Escriu la teva comanda i l'assistent la processarà automàticament.")

    chatbot_ui = gr.Chatbot(label="Xat amb La Parada")
    missatge_input = gr.Textbox(placeholder="Escriu la teva comanda aquí...", label="La teva comanda")
    btn = gr.Button("Enviar comanda", variant="primary")

    def respondre(missatge, historial):
        resposta = processar_comanda(missatge, historial)
        historial.append((missatge, resposta))
        return historial, ""

    btn.click(fn=respondre, inputs=[missatge_input, chatbot_ui], outputs=[chatbot_ui, missatge_input])
    missatge_input.submit(fn=respondre, inputs=[missatge_input, chatbot_ui], outputs=[chatbot_ui, missatge_input])

demo.launch(share=True)

/tmp/ipykernel_5155/1990248457.py:7: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot_ui = gr.Chatbot(label="Xat amb La Parada")
/tmp/ipykernel_5155/1990248457.py:7: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot_ui = gr.Chatbot(label="Xat amb La Parada")


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3250a128d137cb9576.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
